# Исследование набора дневной истории MOEX

## Цель

На фиксированной выборке акций исследовать структуру и качество
дневной истории торгов MOEX и подготовить описание будущей таблицы PostgreSQL.

## Исследуемая выборка

- торговая система: `stock`;
- рынок: `shares`;
- режим торгов: `TQBR`;
- инструменты: `SBER`, `GAZP`, `LKOH`;
- период: июль 2026 года.

## В рамках ноутбука

- загрузить дневную историю выбранных инструментов;
- объединить результаты в единый набор;
- определить гранулярность строки и естественный ключ;
- проверить полноту периода и количество записей;
- изучить столбцы и типы данных;
- проверить дубликаты и пропуски;
- разобраться в назначении полей;
- выбрать поля для будущей таблицы PostgreSQL;
- сформировать предварительную структуру таблицы.

Свечи, сделки, стакан и промышленная загрузка в этом ноутбуке
не рассматриваются.

In [1]:
import pandas as pd
import requests

pd.set_option('display.max_columns', None)

In [ ]:
ENGINE = 'stock'
MARKET = 'shares'
BOARD = 'TQBR'
SECURITES = ["SBER", "GAZP", "LKOH",]

In [ ]:
# 1.загрузить дневную историю для нескольких инструментов
params = {
    "from": "2026-07-01",
    "till": "2026-07-31",
    "lang": "ru"
}

def get_daily_history_frames(url:str, params: dict, company: str)-> pd.DataFrame:
    respons = requests.get(
        url=url,
        params=params,
        timeout=30,
    )
    print(company)
    print(respons.url)
    print(respons.status_code)

    data = respons.json()
    df = pd.DataFrame(
        columns=data['history']['columns'],
        data=data['history']['data']
    )
    return df

daily_history_frames = []
for company in  SECURITES:
    url = f'https://iss.moex.com/iss/history/engines/{ENGINE}/markets/{MARKET}/boards/{BOARD}/securities/{BOARD}.json'
    daily_history_frames.append(get_daily_history_frames(url, params, company))

daily_history_df  = pd.concat(daily_history_frames)
daily_history_df.head(5)

SBER
https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/SBER.json?from=2026-07-01&till=2026-07-31&lang=ru
200
GAZP
https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/GAZP.json?from=2026-07-01&till=2026-07-31&lang=ru
200
LKOH
https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/LKOH.json?from=2026-07-01&till=2026-07-31&lang=ru
200


,BOARDID,TRADEDATE,SHORTNAME,SECID,NUMTRADES,VALUE,OPEN,LOW,HIGH,LEGALCLOSEPRICE,...,MARKETPRICE3,ADMITTEDQUOTE,MP2VALTRD,MARKETPRICE3TRADESVALUE,ADMITTEDVALUE,WAVAL,TRADINGSESSION,CURRENCYID,TRENDCLSPR,TRADE_SESSION_DATE
0,TQBR,2026-07-01,Сбербанк,SBER,181706,1.257712e+10,308.12,305.55,311.57,306.68,...,308.48,None,1.066855e+10,1.066855e+10,None,0,3,SUR,-0.34,2026-07-01
1,TQBR,2026-07-02,Сбербанк,SBER,197020,1.393199e+10,307.10,299.83,308.20,301.80,...,303.30,None,1.229356e+10,1.229356e+10,None,0,3,SUR,-1.95,2026-07-02
2,TQBR,2026-07-03,Сбербанк,SBER,161463,9.917346e+09,301.65,298.04,303.25,301.75,...,300.32,None,8.649960e+09,8.649960e+09,None,0,3,SUR,0.05,2026-07-03
3,TQBR,2026-07-06,Сбербанк,SBER,323901,1.508306e+10,301.39,294.07,305.60,298.68,...,300.12,None,1.187651e+10,1.187651e+10,None,0,3,SUR,-0.80,2026-07-06
4,TQBR,2026-07-07,Сбербанк,SBER,248284,1.459397e+10,299.02,292.12,299.47,296.93,...,295.01,None,1.250731e+10,1.250731e+10,None,0,3,SUR,0.07,2026-07-07


In [ ]:
# 2.Загрузить минутные свечи для несколькизх иснтрументов дял отрабоки пагинации
params_candles = {
    "from": "2026-07-01",
    "till": "2026-07-05",
    "interval": 1,
}

def get_candles_page(url:str, params: dict, company_tiket: str)-> pd.DataFrame:
    response = requests.get(
        url=url,
        params=params,
        timeout=30,
    )

    if response.status_code != 200:
        print(f'Ошибка выгрузки, код: {response.status_code}')

    data = response.json()

    df = pd.DataFrame(
        columns=data['candles']['columns'],
        data=data['candles']['data']
    )
    df['secid'] = company_tiket
    return df

candles_frames = []

for security in SECURITES:
    url = f'https://iss.moex.com/iss/engines/{ENGINE}/markets/{MARKET}/boards/{BOARD}/securities/{security}/candles.json'
    params_candles["start"] = 0

    while True:
        df = get_candles_page(url, params_candles, security)

        if df.empty:
            break
        else:
            params_candles['start'] += df.shape[0]
            candles_frames.append(df)

df_candles = pd.concat(candles_frames, ignore_index=True)
df_candles.shape[0]

12297

In [5]:
df_candles.tail()

,open,close,high,low,value,volume,begin,end,secid
12292,4269.0,4270.0,4270.0,4268.5,230522.0,54,2026-07-05 18:55:00,2026-07-05 18:55:59,LKOH
12293,4270.0,4272.0,4272.0,4268.0,1618612.0,379,2026-07-05 18:56:00,2026-07-05 18:56:59,LKOH
12294,4269.5,4270.0,4272.0,4268.5,367318.0,86,2026-07-05 18:57:00,2026-07-05 18:57:59,LKOH
12295,4271.5,4268.0,4271.5,4268.0,328867.0,77,2026-07-05 18:58:00,2026-07-05 18:58:59,LKOH
12296,4271.5,4271.5,4271.5,4271.5,17086.0,4,2026-07-05 18:59:00,2026-07-05 18:59:59,LKOH


In [6]:
df_candles.duplicated(subset =["begin", "secid"]).sum()

np.int64(0)